In [1]:
import pandas as pd
from pathlib import Path

In [2]:
PROJECT_DIR = Path("/home/sc4r3cr0w/Schreibtisch/Data Analytics/WBS_Final_Project")
RAW = PROJECT_DIR / "RAW"
CLEAN_DIR = PROJECT_DIR / "CLEAN"
CLEAN_DIR.mkdir(exist_ok=True)

KBA_DIR = RAW / "kba_stock"

for folder in [KBA_DIR]:
    print(f"{folder.name:20} exists: {folder.exists()}  files: {len(list(folder.glob('*')))}")

kba_stock            exists: True  files: 11


In [3]:
def find_column(df, pattern):
    """Return the first column that contains the given text pattern."""
    for col in df.columns:
        if df[col].astype(str).str.contains(pattern).any():
            return col
    raise ValueError(f"No column found for pattern: {pattern}")

In [4]:
def read_kba(file, year):
    """Read one KBA FZ1 file (2018 onwards) and return one row per district."""
    sheet = "FZ1.2" if year >= 2020 else "Pkw"
    df = pd.read_excel(file, sheet_name=sheet, header=None)

    district_col = find_column(df, r"^\d{5} ")
    cars_col = find_column(df, "Insgesamt")
    bev_col = find_column(df, "Elektro")

    is_district = df[district_col].astype(str).str.match(r"^\d{5} ")
    districts = df[is_district]

    result = pd.DataFrame({
        "year": year,
        "district_key": districts[district_col].str[:5],
        "district": districts[district_col].str[5:].str.strip(),
        "cars_total": districts[cars_col],
        "bev": districts[bev_col],
    })
    return result.reset_index(drop=True)

In [5]:
df = pd.read_excel(KBA_DIR / "kba_fz1_2026.xlsx", sheet_name="FZ1.2", header=None)

district_col = find_column(df, r"^\d{5} ")
cars_col = find_column(df, "Insgesamt")
bev_col = find_column(df, "Elektro")

print(district_col, cars_col, bev_col)

3 4 10


In [6]:
is_district = df[district_col].astype(str).str.match(r"^\d{5} ")
districts = df[is_district]

print(len(districts))
districts[[district_col, cars_col, bev_col]].head()

400


,3,4,10
9,"08111 STUTTGART,STADT",299197,21837
10,08115 BOEBLINGEN,265915,18590
11,08116 ESSLINGEN,345524,17026
12,08117 GOEPPINGEN,172252,6682
13,08118 LUDWIGSBURG,342859,15642


In [7]:
kba_2026 = pd.DataFrame({
    "year": 2026,
    "district_key": districts[district_col].str[:5],
    "district": districts[district_col].str[5:].str.strip(),
    "cars_total": districts[cars_col],
    "bev": districts[bev_col],
})
kba_2026 = kba_2026.reset_index(drop=True)
kba_2026.head()

,year,district_key,district,cars_total,bev
0,2026,08111,"STUTTGART,STADT",299197,21837
1,2026,08115,BOEBLINGEN,265915,18590
2,2026,08116,ESSLINGEN,345524,17026
3,2026,08117,GOEPPINGEN,172252,6682
4,2026,08118,LUDWIGSBURG,342859,15642


In [8]:
test_2026 = read_kba(KBA_DIR / "kba_fz1_2026.xlsx", 2026)
test_2018 = read_kba(KBA_DIR / "kba_fz1_2018.xlsx", 2018)

print(len(test_2026), test_2026["bev"].sum())
print(len(test_2018), test_2018["bev"].sum())

400 2031870
400 53820


In [9]:
def read_kba_2017(file):
    """Read the 2017 table that was extracted from the KBA PDF (clean header row)."""
    df = pd.read_excel(file, sheet_name="Pkw", dtype={"Kreisschluessel": str})

    result = pd.DataFrame({
        "year": 2017,
        "district_key": df["Kreisschluessel"],
        "district": df["Zulassungsbezirk"],
        "cars_total": df["Pkw_insgesamt"],
        "bev": df["Elektro_BEV"],
    })
    return result

In [10]:
test_2017 = read_kba_2017(KBA_DIR / "kba_fz1_2017.xlsx")
print(len(test_2017), test_2017["bev"].sum())
test_2017.head(3)

400 34008


,year,district_key,district,cars_total,bev
0,2017,08111,"STUTTGART,STADT",298172,814
1,2017,08115,BOEBLINGEN,244396,474
2,2017,08116,ESSLINGEN,319920,325


In [11]:
frames = []

for year in range(2017, 2027):
    file = KBA_DIR / f"kba_fz1_{year}.xlsx"
    if year == 2017:
        df_year = read_kba_2017(file)
    else:
        df_year = read_kba(file, year)
    frames.append(df_year)
    print(year, len(df_year))

kba = pd.concat(frames, ignore_index=True)
print("Total rows:", len(kba))
kba.head()

2017 400
2018 400
2019 400
2020 400
2021 400
2022 399
2023 399
2024 399
2025 399
2026 400
Total rows: 3996


,year,district_key,district,cars_total,bev
0,2017,08111,"STUTTGART,STADT",298172,814
1,2017,08115,BOEBLINGEN,244396,474
2,2017,08116,ESSLINGEN,319920,325
3,2017,08117,GOEPPINGEN,158197,110
4,2017,08118,LUDWIGSBURG,319997,413


In [12]:
keys_2021 = set(kba[kba["year"] == 2021]["district_key"])
keys_2022 = set(kba[kba["year"] == 2022]["district_key"])
keys_2026 = set(kba[kba["year"] == 2026]["district_key"])

print("Only in 2021:", keys_2021 - keys_2022)
print("New in 2026:", keys_2026 - keys_2022)

Only in 2021: {'16056'}
New in 2026: {'06415'}


In [13]:
# 1. data types
print(kba.dtypes)

# 2. missing values
print(kba.isna().sum())

# 3. duplicates: each district may appear only once per year
print("Duplicates:", kba.duplicated(subset=["year", "district_key"]).sum())

year             int64
district_key       str
district           str
cars_total      object
bev             object
dtype: object
year            0
district_key    0
district        0
cars_total      0
bev             0
dtype: int64
Duplicates: 0


In [14]:
# chaning cars_total & bev type data type from object to integer
kba["cars_total"] = pd.to_numeric(kba["cars_total"]).astype(int)
kba["bev"] = pd.to_numeric(kba["bev"]).astype(int)

print(kba.dtypes)

year            int64
district_key      str
district          str
cars_total      int64
bev             int64
dtype: object


In [15]:
yearly = kba.groupby("year")[["cars_total", "bev"]].sum()
yearly["bev_share_pct"] = (yearly["bev"] / yearly["cars_total"] * 100).round(2)
yearly

,cars_total,bev,bev_share_pct
year,,,
2017,45782167,34008,0.07
2018,46455203,53820,0.12
2019,47077855,83125,0.18
2020,47699065,136546,0.29
2021,48207878,308745,0.64
2022,48498569,617846,1.27
2023,48720180,1011996,2.08
2024,49054268,1407224,2.87
2025,49291479,1649725,3.35


In [16]:
kba.to_csv(CLEAN_DIR / "kba_stock.csv", index=False)
print("Saved:", CLEAN_DIR / "kba_stock.csv")

Saved: /home/sc4r3cr0w/Schreibtisch/Data Analytics/WBS_Final_Project/CLEAN/kba_stock.csv
